# Chapter 07: Data Engineering for AI Systems

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abda-dc/ai-engineering-zero-to-hero/blob/main/book/chapters/part3-rag-data/ch07-data-engineering-for-ai.ipynb)

**Part:** Part III: Vector Databases, Data & Production RAG  
**Chapter Identifier:** `ch07`  

## Core Concepts Covered in this Chapter:
- **Data Collection/APIs**
- **ETL/ELT Pipelines**
- **Data Cleaning**
- **SQL/NoSQL**
- **Streaming**
- **Feature Stores**
- **Data Versioning with DVC**

---


## 1. Modern AI Data Pipelines: ETL vs. ELT

Data is the lifeblood of AI. While traditional ETL transforms data before loading into warehouses, modern AI pipelines often employ ELT or streaming ingestion to preserve raw unstructured data in data lakes for continuous embedding generation and re-indexing.

```mermaid
flowchart LR
    A[Data Sources: APIs, DBs, Docs] --> B[Ingestion Queue / Kafka]
    B --> C[Data Lake Raw Storage]
    C --> D[Cleaning & Deduplication]
    D --> E[Chunking & Feature Store]
    E --> F[Vector DB & Semantic Cache]
```


In [ ]:
import hashlib

def deduplicate_documents(docs: list) -> list:
    seen_hashes = set()
    unique_docs = []
    for doc in docs:
        content_hash = hashlib.sha256(doc.strip().lower().encode('utf-8')).hexdigest()
        if content_hash not in seen_hashes:
            seen_hashes.add(content_hash)
            unique_docs.append(doc)
    return unique_docs

raw_stream = [
    "Machine Learning is changing software engineering.",
    "Data engineering powers AI systems.",
    "Machine Learning is changing software engineering.",
    "data engineering powers ai systems."
]

deduped = deduplicate_documents(raw_stream)
print(f"Original: {len(raw_stream)} docs -> Deduped: {len(deduped)} docs")

## 2. Text Cleaning & Normalization for Embeddings

Garbage in, garbage out. Unstructured data often contains HTML tags, whitespace noise, non-printable characters, and boilerplate headers that degrade vector embedding fidelity.


In [ ]:
import re

def clean_text_for_embedding(raw_html: str) -> str:
    cleaned = re.sub(r'<[^>]+>', ' ', raw_html)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', cleaned)
    return cleaned.strip()

sample_html = """<div>
    <h1>AI Architecture &amp; Design</h1>
    <p>Retrieval-Augmented Generation (RAG) \t connects models to live data.\n\n</p>
</div>"""

print("Cleaned Output:\n", clean_text_for_embedding(sample_html))

## 3. Feature Stores & Data Versioning (DVC)

Feature Stores provide a centralized repository for both batch and real-time features. Data Version Control (DVC) tracks dataset and model checkpoints using Git-like commit pointers without bloating code repositories.


In [ ]:
class ToyFeatureStore:
    def __init__(self):
        self.online_features = {}

    def set_entity_features(self, entity_id: str, features: dict):
        self.online_features[entity_id] = features

    def get_entity_features(self, entity_id: str) -> dict:
        return self.online_features.get(entity_id, {"error": "Entity not found"})

fs = ToyFeatureStore()
fs.set_entity_features("user_901", {"tier": "enterprise", "preferred_model": "claude-3-5-sonnet", "rag_context_limit": 8192})
print("Online Feature Lookup:", fs.get_entity_features("user_901"))

---
## Hands-On Capstone Lab: Build an Automated Ingestion, Cleaning & Versioning Pipeline

### Objective:
Build a complete pipeline that ingests raw documents, deduplicates them, cleans text, extracts metadata features, and registers a dataset manifest.

### Instructions:
1. Ingest a batch of noisy document records.
2. Clean and deduplicate.
3. Compute dataset version hash (manifest signature) and register.


In [ ]:
def run_ai_data_pipeline(records: list) -> dict:
    cleaned_records = []
    seen = set()
    for rec in records:
        cleaned_body = clean_text_for_embedding(rec["text"])
        chash = hashlib.md5(cleaned_body.encode('utf-8')).hexdigest()
        if chash in seen:
            continue
        seen.add(chash)
        cleaned_records.append({
            "id": rec["id"],
            "hash": chash,
            "text": cleaned_body,
            "word_count": len(cleaned_body.split()),
            "source": rec.get("source", "unknown")
        })
    manifest_id = hashlib.sha256(str([r["hash"] for r in cleaned_records]).encode('utf-8')).hexdigest()[:12]
    return {
        "manifest_version": f"dvc-v{manifest_id}",
        "record_count": len(cleaned_records),
        "records": cleaned_records
    }

raw_batch = [
    {"id": "doc_1", "text": "<h3>Introduction to AI</h3><p>AI is transforming industries.</p>", "source": "wiki"},
    {"id": "doc_2", "text": "<p>AI is transforming industries.</p>", "source": "blog"},
    {"id": "doc_3", "text": "<div>Vector databases store high-dimensional embeddings.</div>", "source": "docs"}
]

manifest = run_ai_data_pipeline(raw_batch)
print(f"Dataset Manifest: {manifest['manifest_version']} with {manifest['record_count']} unique records")
for r in manifest["records"]:
    print(f"  [{r['id']}] Words: {r['word_count']} -> '{r['text']}'")

---
## Chapter Mastery Self-Check Quiz

Test your understanding of the concepts covered in this chapter.

### Question 1: What is the primary role of Data Version Control (DVC) in AI engineering?
- **A)** To replace Git entirely
- **B)** To version large datasets, embeddings, and model weights via small metadata pointer files tracked in Git
- **C)** To generate synthetic text automatically
- **D)** To speed up GPU training by 10x

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) To version large datasets, embeddings, and model weights via small metadata pointer files tracked in Git

**Explanation:** DVC manages large data files outside the Git working tree while tracking version hash pointers in Git, ensuring full reproducibility without repository bloat.
</details>

### Question 2: Why is text cleaning (e.g., stripping HTML tags and control characters) critical before generating embeddings?
- **A)** HTML tags consume embedding token budget and introduce irrelevant syntactic noise into the semantic vector representation
- **B)** Embedding models cannot process any text containing the letter '<'
- **C)** It compresses text to 10% of its size
- **D)** It guarantees 100% classification accuracy

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) HTML tags consume embedding token budget and introduce irrelevant syntactic noise into the semantic vector representation

**Explanation:** Formatting boilerplate and markup noise consume valuable context tokens and distort vector embeddings away from the underlying semantic meaning.
</details>

### Question 3: What is the key difference between an Online Feature Store and an Offline Feature Store?
- **A)** Online stores provide low-latency single-record lookups for real-time inference; offline stores provide high-throughput batch queries for training
- **B)** Online stores use paper printouts; offline stores use cloud databases
- **C)** Offline stores are only accessible without an internet connection
- **D)** There is no difference

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** A) Online stores provide low-latency single-record lookups for real-time inference; offline stores provide high-throughput batch queries for training

**Explanation:** Online stores (often backed by Redis or Cassandra) serve real-time predictions with sub-millisecond latency, while offline stores (Parquet/Snowflake) serve bulk training data.
</details>

### Question 4: In data pipeline design, what does idempotency mean?
- **A)** A pipeline that runs at 10x speed
- **B)** A pipeline that produces the identical state regardless of how many times it is re-executed with the same input
- **C)** A pipeline that deletes all data upon error
- **D)** A pipeline that uses only Python code

<details>
<summary><b>View Answer & Explanation</b></summary>

**Correct Answer:** B) A pipeline that produces the identical state regardless of how many times it is re-executed with the same input

**Explanation:** Idempotent operations can be applied multiple times without changing the result beyond the initial application, critical for fault-tolerant data pipelines.
</details>
